In [3]:
import os, json
from tqdm import tqdm
from collections import defaultdict

In [15]:
# JSON 파일이 들어있는 폴더 경로
input_dir = "C:/Users/User/Desktop/datasets/lane_and_crosswalk/Validation/labels"
# 필터링된 JSON을 저장할 폴더 경로
output_dir = "C:/Users/User/Desktop/datasets/lane_and_crosswalk/Validation/clean_labels"
# 이미지 폴더
images_dir = "C:/Users/User/Desktop/datasets/lane_and_crosswalk/Validation/images"


In [11]:
# 어노테이션 비어있는 파일 기록용
empty_annotation_files = []

# input_dir 기준으로 필터링 작업 수행
json_files = [f for f in os.listdir(input_dir) if f.endswith(".json")]

for json_file in tqdm(json_files, desc="Processing JSON files"):
    input_path = os.path.join(input_dir, json_file)
    output_path = os.path.join(output_dir, json_file)

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    annotations = data.get("annotations", [])

    if not annotations:
        empty_annotation_files.append(json_file)
        with open(output_path, "w", encoding="utf-8") as f_out:
            json.dump({
                "image": data.get("image", {}),
                "annotations": []
            }, f_out, ensure_ascii=False, indent=2)
        continue

    filtered_annotations = []

    for annotation in annotations:
        cls = annotation.get("class")

        if cls == "crosswalk":
            filtered_annotations.append(annotation)

        elif cls == "traffic_lane":
            attributes = annotation.get("attributes", [])
            color = None
            for attr in attributes:
                if attr.get("code") == "lane_color":
                    color = attr.get("value")
                    break
            if color == "yellow":
                filtered_annotations.append(annotation)

    # 저장
    filtered_data = {
        "image": data.get("image", {}),
        "annotations": filtered_annotations
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, ensure_ascii=False, indent=2)

# === 이제 output_dir 기준으로 class 및 속성 개수 통계 ===

class_counter = defaultdict(int)
lane_color_counter = defaultdict(int)

output_json_files = [f for f in os.listdir(output_dir) if f.endswith(".json")]

for json_file in tqdm(output_json_files, desc="Counting classes (output_dir 기준)"):
    output_path = os.path.join(output_dir, json_file)

    with open(output_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    annotations = data.get("annotations", [])
    for annotation in annotations:
        cls = annotation.get("class")
        class_counter[cls] += 1

        if cls == "traffic_lane":
            for attr in annotation.get("attributes", []):
                if attr.get("code") == "lane_color":
                    color = attr.get("value")
                    lane_color_counter[color] += 1
                    break

# 결과 출력
print("\n===== [output_dir 기준] 클래스별 개수 =====")
for cls, count in class_counter.items():
    print(f"{cls}: {count}개")

print("\n===== [output_dir 기준] traffic_lane - lane_color 별 개수 =====")
for color, count in lane_color_counter.items():
    print(f"{color}: {count}개")

print("\n===== 어노테이션이 비어있는 파일 (input 기준 필터링 결과) =====")
print(f"총 {len(empty_annotation_files)}개 파일에서 어노테이션이 없습니다.")
if empty_annotation_files:
    print("파일 목록:")
    for filename in empty_annotation_files:
        print(f" - {filename}")


Counting classes (output_dir 기준): 100%|██████████| 30277/30277 [02:01<00:00, 249.39it/s]


===== [output_dir 기준] 클래스별 개수 =====
traffic_lane: 60437개
crosswalk: 21595개

===== [output_dir 기준] traffic_lane - lane_color 별 개수 =====
yellow: 60437개

===== 어노테이션이 비어있는 파일 (input 기준 필터링 결과) =====
총 512개 파일에서 어노테이션이 없습니다.
파일 목록:
 - 15229769.json
 - 15229770.json
 - 15229771.json
 - 15229772.json
 - 15229973.json
 - 15229974.json
 - 15229975.json
 - 15229976.json
 - 15229979.json
 - 15230005.json
 - 15230006.json
 - 15230008.json
 - 15230011.json
 - 15230087.json
 - 15230088.json
 - 15230090.json
 - 15230091.json
 - 15230093.json
 - 15230156.json
 - 15230157.json
 - 15230166.json
 - 15233744.json
 - 15233745.json
 - 15233746.json
 - 15233747.json
 - 15233748.json
 - 15233749.json
 - 15233750.json
 - 15233751.json
 - 15233752.json
 - 15233753.json
 - 15233754.json
 - 15233755.json
 - 15233756.json
 - 15233757.json
 - 15233758.json
 - 15235309.json
 - 15235310.json
 - 15235311.json
 - 15235328.json
 - 15235329.json
 - 15235330.json
 - 15235333.json
 - 15235334.json
 - 15235339.json
 - 152

In [12]:
# === 3차: 어노테이션 비어있는 JSON + 이미지(jpg) 삭제 ===

for json_file in tqdm(empty_annotation_files, desc="Deleting empty annotation files"):
    json_path = os.path.join(output_dir, json_file)
    if os.path.exists(json_path):
        os.remove(json_path)

    # 이미지 파일 삭제 (.jpg 고정)
    base_name = os.path.splitext(json_file)[0]
    image_path = os.path.join(images_dir, base_name + ".jpg")
    if os.path.exists(image_path):
        os.remove(image_path)


Deleting empty annotation files: 100%|██████████| 512/512 [00:00<00:00, 2628.83it/s]


In [13]:
# JSON 파일 이름(확장자 제외)
json_files = [os.path.splitext(f)[0] for f in os.listdir(output_dir) if f.endswith(".json")]

# 이미지 파일 이름(확장자 제외)
image_files = [os.path.splitext(f)[0] for f in os.listdir(images_dir) if f.lower().endswith(".jpg")]

# 집합으로 변환
json_set = set(json_files)
image_set = set(image_files)

# 누락 확인
json_only = json_set - image_set
image_only = image_set - json_set

# 결과 출력
print(f"✅ JSON 파일 수: {len(json_set)}")
print(f"✅ 이미지 파일 수: {len(image_set)}")

if not json_only and not image_only:
    print("🎉 JSON과 이미지 파일이 완전히 일치합니다.")
else:
    print("⚠ 일치하지 않는 파일이 있습니다:")

    if json_only:
        print(f"\n❌ 이미지가 없는 JSON 파일 ({len(json_only)}개):")
        for name in sorted(json_only):
            print(f" - {name}.json")

    if image_only:
        print(f"\n❌ JSON이 없는 이미지 파일 ({len(image_only)}개):")
        for name in sorted(image_only):
            print(f" - {name}.jpg")


✅ JSON 파일 수: 29765
✅ 이미지 파일 수: 29765
🎉 JSON과 이미지 파일이 완전히 일치합니다.


In [ ]:
# 카운터 초기화
class_counter = defaultdict(int)
lane_color_yellow = 0
lane_color_not_yellow = 0
lane_type_solid = 0
lane_type_not_solid = 0

# JSON 파일 리스트
json_files = [f for f in os.listdir(output_dir) if f.endswith(".json")]

for json_file in tqdm(json_files, desc="분석 중"):
    path = os.path.join(output_dir, json_file)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    annotations = data.get("annotations", [])

    for ann in annotations:
        cls = ann.get("class")
        class_counter[cls] += 1

        if cls == "traffic_lane":
            attributes = ann.get("attributes", [])

            color = None
            type_ = None

            for attr in attributes:
                if attr.get("code") == "lane_color":
                    color = attr.get("value")
                elif attr.get("code") == "lane_type":
                    type_ = attr.get("value")

            # 색상 카운트
            if color == "yellow":
                lane_color_yellow += 1
            else:
                lane_color_not_yellow += 1

            # 타입 카운트
            if type_ == "solid":
                lane_type_solid += 1
            else:
                lane_type_not_solid += 1

# 결과 출력
print("\n===== 클래스별 개수 =====")
for cls, count in class_counter.items():
    print(f"{cls}: {count}개")

print("\n===== traffic_lane 속성 분석 =====")
print(f"차선 중 황색선: {lane_color_yellow}개")
print(f"다른 색: {lane_color_not_yellow}개")
print(f"차선 중 실선: {lane_type_solid}개")
print(f"차선 중 점선: {lane_type_not_solid}개")


분석 중: 100%|██████████| 29765/29765 [00:02<00:00, 11348.77it/s]


===== 클래스별 개수 =====
traffic_lane: 60437개
crosswalk: 21595개

===== traffic_lane 속성 분석 =====
lane_color == yellow: 60437개
lane_color != yellow: 0개
lane_type == solid: 54274개
lane_type != solid: 6163개
